## Fluidic pinball test case

In [1]:
# IMPORT UTILITIES

import random
import numpy as np
import torch

random.seed(0)
np.random.seed(0)
torch.manual_seed(0);

In [2]:
# PROBLEM DEFINITION

from utils.pinball import Pinball

pinball = Pinball(diffusion = 0.001, mu = 1.0, beta = 0.0001, beta_g = 10, dt = 1.0, T = 50.0)

In [ ]:
# SOLUTION EXAMPLE

pinball_velocities = [0.5, 0.5, 1.0]

yt, v, J = pinball.solve_FP(pinball_velocities)

In [ ]:
# STATE VISUALIZATION

from utils.postprocessing import trajectory

trajectory(yt, pinball.plot_state, title = "State", times = pinball.times)

## Data loading

In [ ]:
# DATA GENERATION (SKIP THIS CELL IF DATA ALREADY AVAILABLE)

ntrajectories = 100
parameters_ranges = [(-1.0, 1.0), (-1.0, 1.0), (-1.0, 1.0)] # Parameters: cylinder velocities
nparameters = len(parameters_ranges)

Y = np.empty((ntrajectories, pinball.ntimesteps, pinball.nstate))
U = np.empty((ntrajectories, pinball.ntimesteps-1, pinball.nvelocity))
V = np.empty((ntrajectories, pinball.nvelocity))
MU = np.empty((ntrajectories, pinball.ntimesteps, nparameters))

for i in range(ntrajectories):

    print(f"Generating snapshots n.{i}...")
    
    # generate random parameters
    parameters = [0] * nparameters
    for j in range(len(parameters_ranges)):
        parameters[j] = (parameters_ranges[j][1] - parameters_ranges[j][0]) * np.random.rand() + parameters_ranges[j][0]    
    
    # solve optimal control problem
    yt_opt, ut_opt, v, J_opt = pinball.solve_OCP(parameters)
    
    # store results
    Y[i] = yt_opt
    U[i] = ut_opt
    V[i] = v
    MU[i] = parameters
    
    np.savez("data/Pinball/pinball.npz", state = Y, control = U, velocity = V, parameters = MU)

print("Snapshots generated!")

In [3]:
# LOAD SNAPSHOTS MATRICES

from utils.datamanager import DataManager

parameters_ranges = [(-1.0, 1.0), (-1.0, 1.0), (-1.0, 1.0)] # Parameters: cylinder velocities
nparameters = len(parameters_ranges)

datamanager = DataManager(np.load("data/Pinball/pinball.npz"),
                          dims = {'state': 1, 'control': 2, 'velocity': 2},
                          train_ratio = 0.8,
                          valid_ratio = 0.1,
                          test_ratio = 0.1)

In [ ]:
# STATE VISUALIZATION

from utils.postprocessing import trajectories

which_trajectories = [0, 1, 2]

trajectories([datamanager.data["state"][which_trajectories[0]], datamanager.data["state"][which_trajectories[1]], datamanager.data["state"][which_trajectories[2]]],
             pinball.plot_state,
             titles = ["First scenario", "Second scenario", "Third scenario"],
             times = pinball.times,
             figsize = (15, 5))

In [ ]:
# CONTROL VISUALIZATION

which_trajectories = [0, 1, 2]

trajectories([datamanager.data["control"][which_trajectories[0]], datamanager.data["control"][which_trajectories[1]], datamanager.data["control"][which_trajectories[2]]],
             pinball.plot_control,
             titles = ["First scenario", "Second scenario", "Third scenario"],
             times = pinball.times,
             figsize = (15, 5))

In [5]:
# TRAIN-VALIDATION-TEST SPLITTING

datamanager.prepare()

## Data compression

In [6]:
# PROPER ORTHOGONAL DECOMPOSITION (POD)

kcontrol = 300
datamanager.POD(ranks = {'control': kcontrol})

Field: control, Component: 0, Test MRE: 3.59%
Field: control, Component: 1, Test MRE: 3.59%


# SHRED-ROM

In [ ]:
# EXTRACT SENSOR COORDINATES AND DATA (SKIP THIS CELL IF COORDINATES ALREADY AVAILABLE)

from fenics import Function

nsensors = 1
ntrajectories = datamanager.ntrajectories
ntimesteps = datamanager.data["state"].shape[1]
sensor_coordinates = torch.zeros(datamanager.ntrajectories, ntimesteps, 2)
initial_coordinates = torch.zeros(2)

for i in range(ntrajectories):
    v = Function(pinball.Vh)
    v.vector()[:] = datamanager.data["velocity"][i]
    v.set_allow_extrapolation(True)
    for j in range(ntimesteps):
        if j == 0:
            sensor_coordinates[i,j] = initial_coordinates
        else:
            sensor_coordinates[i,j] = sensor_coordinates[i,j-1] + pinball.dt * v(sensor_coordinates[i,j-1])

sensor_coordinates_train = sensor_coordinates[datamanager.train]
sensor_coordinates_valid = sensor_coordinates[datamanager.valid]
sensor_coordinates_test = sensor_coordinates[datamanager.test]

In [8]:
# LOAD SENSOR DATA

nsensors = 1
sensor_coordinates = torch.load('data/Pinball/sensors.pt', weights_only = True)

sensor_coordinates_train = sensor_coordinates[datamanager.train]
sensor_coordinates_valid = sensor_coordinates[datamanager.valid]
sensor_coordinates_test = sensor_coordinates[datamanager.test]

In [9]:
# BUILD TRAIN, VALIDATION AND TEST DATASETS WITH PADDING

from utils.datamanager import Padding, TimeSeriesDataset

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

lag = 10

train_data_in = Padding(sensor_coordinates_train, lag).to(device)
valid_data_in = Padding(sensor_coordinates_valid, lag).to(device)
test_data_in = Padding(sensor_coordinates_test, lag).to(device)

train_data_out = Padding(torch.from_numpy(datamanager.data_POD["control"][datamanager.train,:,:]), 1).squeeze(1).to(device)
valid_data_out = Padding(torch.from_numpy(datamanager.data_POD["control"][datamanager.valid,:,:]), 1).squeeze(1).to(device)
test_data_out = Padding(torch.from_numpy(datamanager.data_POD["control"][datamanager.test,:,:]), 1).squeeze(1).to(device)

train_dataset = TimeSeriesDataset(train_data_in, train_data_out)
valid_dataset = TimeSeriesDataset(valid_data_in, valid_data_out)
test_dataset = TimeSeriesDataset(test_data_in, test_data_out)

In [ ]:
# SHRED TRAINING (SKIP THIS CELL IF MODEL ALREADY AVAILABLE)

from utils.models import SHRED, fit

shred = SHRED(2 * nsensors, kcontrol, hidden_size = 64, hidden_layers = 2, decoder_sizes = [350, 400], dropout = 0.1).to(device)
train_errors, valid_errors = fit(shred, train_dataset, valid_dataset, batch_size = 64, epochs = 500, lr = 1e-3, verbose = True, patience = 100)
train_errors, valid_errors = fit(shred, train_dataset, valid_dataset, batch_size = 64, epochs = 500, lr = 1e-4, verbose = True, patience = 100)

In [10]:
# LOAD PRE-TRAINED SHRED MODEL

from utils.models import SHRED

shred = SHRED(2 * nsensors, kcontrol, hidden_size = 64, hidden_layers = 2, decoder_sizes = [350, 400], dropout = 0.1).to(device)
shred.load_state_dict(torch.load('data/Pinball/shred.pt', weights_only = False, map_location = torch.device('cpu')));

In [11]:
# SHRED RECONSTRUCTIONS

from utils.postprocessing import mre_numpy, num2p

shred.freeze()

shred_pred = shred(test_data_in).cpu().numpy()
shred_pred = shred_pred.reshape(datamanager.ntest, -1, kcontrol)

data_test_pred = datamanager.decode(data_POD = {'control': shred_pred})

In [ ]:
# GROUND TRUTH vs SHRED-ROM PREDICTION

which_trajectory = 0

trajectories([datamanager.data["control"][datamanager.test][which_trajectory],
              data_test_pred['control'][which_trajectory],
              np.sqrt((datamanager.data["control"][datamanager.test][which_trajectory] - data_test_pred['control'][which_trajectory])**2)],
              pinball.plot_control,
              titles = ["Ground truth", "SHRED-ROM prediction", "Prediction error"],
              times = pinball.times,
              figsize = (15, 5))

## SHRED-ROM into play

In [ ]:
## PINBALL CONTROL FOR TEST PARAMETERS

from fenics import Function
from IPython.display import clear_output as clc

which_trajectory = 0

v = Function(pinball.Vh)
v.vector()[:] = datamanager.data["velocity"][datamanager.test][which_trajectory]

ut = []
for j in range(pinball.ntimesteps-1):
    u = Function(pinball.Vh)
    u.vector()[:] = data_test_pred["control"][which_trajectory, j]
    ut.append(u)

yt, v, J = pinball.solve_FP(datamanager.data["parameters"][datamanager.test][which_trajectory, 0, :], ut = ut)

clc()
trajectory(yt, pinball.plot_state, title = "State", times = pinball.times)